# Projeto — Análise de Preços de Imóveis

**Dataset:** Housing Prices Dataset (King County)

Notebook organizado segundo as fases e questões do desafio.

## 🔹 FASE 1 — Limpeza e Padronização

### 1. Consolidação e Tipagem
Conversão das colunas para os tipos adequados: data para datetime, áreas de pés quadrados para metros quadrados e código postal para texto.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

#import plotly.io as pio
#pio.renderers.default = "vscode"

In [ ]:
df = pd.read_csv('Housing.csv')

In [ ]:
df['date'] = pd.to_datetime(df['date'])

In [ ]:
#Convertendo a Para Metros Quadrados

converter = ['sqft_living' , 'sqft_lot', 'sqft_above', 'sqft_basement', 'sqft_living15', 'sqft_lot15']

for i in converter:
    df[i]= df[i] * 0.092903

In [ ]:
df['zipcode'] = df['zipcode'].astype(str)

In [ ]:
df.info()

### 2. Tratamento de Valores Ausentes
Verificação de valores nulos por coluna. O dataset não apresenta ausências, portanto nenhum registro precisou ser removido.

In [ ]:
df.isna().sum()

### 3. Padronização de Variáveis Categóricas
Renomeação das colunas para português, padronizando a nomenclatura das variáveis de localização e tipo do imóvel.

In [ ]:

traducao_colunas = {
    'id': 'id_imovel',
    'date': 'data',
    'price': 'preco',
    'bedrooms': 'quartos',
    'bathrooms': 'banheiros',
    'sqft_living': 'm2_area_habitavel',    
    'sqft_lot': 'm2_terreno',
    'floors': 'andares',
    'waterfront': 'vista_mar_rio',         
    'view': 'nota_vista',                  
    'condition': 'condicao_imovel',        
    'grade': 'nota_design',                
    'sqft_above': 'm2_acima_solo',          
    'sqft_basement': 'm2_porao',          
    'yr_built': 'ano_construcao',
    'yr_renovated': 'ano_renovacao',
    'zipcode': 'codigo_postal',
    'lat': 'latitude',
    'long': 'longitude',
    'sqft_living15': 'm2_area_vizinhos_15', 
    'sqft_lot15': 'm2_terreno_vizinhos_15' 
}

df_traduzido = df.rename(columns=traducao_colunas)

print(df_traduzido.columns)

## 🔹 FASE 2 — Consultas e Agregações

### 4. Análise Histórica de Preços
Preço médio por ano e por mês. O período do dataset (mai/2014 a mai/2015) cobre apenas 13 meses, então a comparação anual reflete meses parciais.

In [ ]:
df_traduzido['ano'] = df_traduzido['data'].dt.year
df_traduzido['mes'] = df_traduzido['data'].dt.month

In [ ]:
df_media_anual = df_traduzido.groupby('ano')['preco'].mean().reset_index().round(2)

In [ ]:
df_media_anual

In [ ]:
media_mensal = df_traduzido.groupby(['ano','mes'])['preco'].mean().reset_index().round(2)
media_mensal

### 5. Análise Localizada
Preço médio, mínimo e máximo de um bairro (CEP) específico dentro de um intervalo de anos definido.

In [ ]:
df_traduzido['codigo_postal'].value_counts().reset_index().head()

In [ ]:
bairro_escolhido ='98103'
#intevalo de anos:
ano_ini , ano_fim = 2014, 2015


df_bairro = df_traduzido[(df_traduzido['codigo_postal'] == bairro_escolhido) & (df_traduzido['ano'].between(ano_ini, ano_fim)) ]

print(f'O preço médio do cep {bairro_escolhido} é R${df_bairro['preco'].mean():.2f}, o maior preco é {df_bairro['preco'].max():.2f} o menor é {df_bairro['preco'].min()} ')

### 6. Ranking de Bairros
Cinco bairros com maior preço médio. Os valores mais altos concentram-se em regiões nobres de King County (ex.: 98039 - Medina).

In [ ]:
media_preco_bairro = (df_traduzido.groupby('codigo_postal')['preco'].mean()
.sort_values(ascending=False)
.reset_index()
.rename(columns= ({'preco' : 'Media_preco','codigo_postal' :'Bairro' }))
.round(2)
)
media_preco_bairro


In [ ]:
top5 = media_preco_bairro.head(5)
top5

In [ ]:
fig_top5 = px.bar(
    top5,
    x = 'Media_preco',
    y = 'Bairro',
    orientation='h',
    text_auto = '.3s',
    color = 'Media_preco',
    title ='Top 5 bairros com maiores preços médios'
)
fig_top5.update_yaxes(type = 'category')
fig_top5.show()


## 🔹 FASE 3 — Estatística e Outliers

### 7. Identificação de Outliers
Boxplot e histograma da distribuição de preços. Os valores atípicos correspondem a imóveis de luxo reais (mansões, frente-mar), não a inconsistências, e por isso foram mantidos.

In [ ]:
fig_outlier= px.box(df_traduzido, y='preco' , title= 'Distribuição de Preços com Outliers', points ='outliers')
fig_outlier.show()


In [ ]:
fig_hist = px.histogram(df_traduzido , x = 'preco' , nbins= 100 , title ='Distribuição de Preços')
fig_hist.show()

## 🔹 FASE 4 — Visualização e Análise Exploratória

### 8. Relação entre Área e Preço
Dispersão entre área habitável e preço, evidenciando a tendência de crescimento (não perfeitamente linear).

In [ ]:

fig_area = px.scatter(
    df_traduzido, x="m2_area_habitavel", y="preco", title="Relação preço por área"
)
fig_area.show()

### 9. Evolução Temporal dos Preços
Preço médio mês a mês ao longo de todo o período disponível.

In [ ]:

media_mensal['data_formatada'] = media_mensal['ano'].astype(str) + '-' + media_mensal['mes'].astype(str)

fig_preco = px.line(
    media_mensal,
    x='data_formatada',
    y='preco',
    title='Evolução do Preço ao Longo do Tempo',
    labels={'data_formatada': 'Período (Ano-Mês)', 'preco': 'Preço ($)'}
)
fig_preco.update_xaxes(type='category')

fig_preco.show()

## 🔹 FASE 5 — Engenharia de Atributos e Correlações

### 11. Criação de Novas Variáveis
Atributos derivados: densidade de ocupação do lote, idade efetiva (considerando reformas), total de cômodos e indicador de renovação.

In [ ]:
df_traduzido['habitavel_por_terreno'] = df_traduzido['m2_area_habitavel'] / df_traduzido['m2_terreno']
df_traduzido['idade_efetiva'] = df_traduzido['ano'] - df_traduzido[['ano_construcao','ano_renovacao']].max(axis=1)
df_traduzido['total_comodos'] = df_traduzido['quartos'] + df_traduzido['banheiros']
df_traduzido['foi_renovado'] = (df_traduzido['ano_renovacao'] > 0).astype(int)




### 12. Análise de Correlação
Correlação de Pearson (relações lineares e redundância entre features) comparada com Spearman (relações monotônicas), para identificar as variáveis mais influentes e evitar descartar relações não lineares.

In [ ]:
cols_tirar = ['id_imovel', 'data' , 'codigo_postal']
df_correlacao_pearson = df_traduzido.drop(columns= cols_tirar).corr('pearson')


df_correlacao_pearson

In [ ]:
corr_preco_pearson = df_correlacao_pearson['preco'].drop('preco').abs()
corr_preco_spearman = df_traduzido.corr('spearman')['preco'].sort_values(ascending=False).drop('preco').abs()


In [ ]:
comparacao = pd.DataFrame({
    'pearson': corr_preco_pearson,
    'spearman': corr_preco_spearman
})
comparacao['diferenca'] = (comparacao['spearman'] - comparacao['pearson']).round(3)
comparacao['pearson'] = comparacao['pearson'].round(3)
comparacao['spearman'] = comparacao['spearman'].round(3)

comparacao = comparacao.reindex(
    comparacao['spearman'].abs().sort_values(ascending=False).index
)
print(comparacao)

In [ ]:
fig_heatmap = px.imshow(
    df_correlacao_pearson,
    text_auto='.2f',            # escreve o valor em cada célula
    aspect='auto',
    color_continuous_scale='RdBu_r',   # vermelho = +, azul = -
    zmin=-1, zmax=1,            # fixa a escala de cor de -1 a 1
    title='Mapa de calor — correlação de Pearson entre variáveis'
                        )
fig_heatmap.update_layout(height=800, width=900)
fig_heatmap.show()

### 13. Transformação de Variáveis Categóricas
A única categórica relevante para a modelagem seria o código postal (70 níveis, alta cardinalidade). Optou-se por descartá-la, pois a informação de localização é preservada de forma contínua pelas variáveis **latitude** e **longitude** — evitando a explosão dimensional que um one-hot encoding de 70 colunas causaria. As demais variáveis já são numéricas, dispensando codificação.

## 🔹 FASE 6 — Machine Learning

### 14. Modelo de Previsão de Preço

Versão profissional utilizando as bibliotecas do scikit-learn (`Pipeline`, `StandardScaler`, `PolynomialFeatures`, `Ridge`). O pipeline encadeia normalização, geração de termos polinomiais e regressão regularizada, aplicando as transformações apenas com estatísticas do treino para evitar vazamento de dados.

#### Preparação dos dados
Remoção de duplicatas de revenda (evita vazamento entre treino e teste) e separação treino/teste. As features incluem a geografia (latitude e longitude), já que os experimentos anteriores mostraram seu impacto positivo.

In [ ]:
from sklearn.model_selection import train_test_split

features = [
    'm2_area_habitavel', 'nota_design', 'm2_area_vizinhos_15',
    'total_comodos', 'nota_vista', 'm2_porao',
    'habitavel_por_terreno', 'idade_efetiva', 'vista_mar_rio',
    'latitude', 'longitude'
]

# Remove revendas do mesmo imovel (mantem a venda mais recente)
df_modelo = df_traduzido.sort_values('data').drop_duplicates('id_imovel', keep='last')

X = df_modelo[features]
y = df_modelo['preco']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Treino: {X_train.shape[0]} imoveis | Teste: {X_test.shape[0]}')
print(f'Features ({len(features)}):', features)